In [1]:
from abc import ABC, abstractmethod
from pathlib import Path

import numpy as np
import requests

In [2]:
np.random.seed(42)

In [3]:
class Tensor:

    def __init__(self, data):
        self.data = np.array(data)
        self.grad = np.zeros_like(self.data)
        self.gradient_fn = None
        self.parents = set()

    def backward(self):
        if self.gradient_fn is not None:
            self.gradient_fn()

        for p in self.parents:
            p.backward()

    def __str__(self):
        return f'Tensor({self.data})'

In [4]:
class Dataset(ABC):

    def __init__(self, batch_size=1):
        self.batch_size = batch_size
        self.load()
        self.train()

    @abstractmethod
    def load(self):
        pass

    def train(self):
        self.data = self.train_data

    def eval(self):
        self.data = self.test_data

    def all(self):
        x, y = self.data
        return Tensor(x), Tensor(y)

    def __len__(self):
        x, *_ = self.data
        return len(x) // self.batch_size

    def __getitem__(self, index):
        s = slice(index * self.batch_size, (index + 1) * self.batch_size)
        x, y = self.data
        return Tensor(x[s]), Tensor(y[s])

In [5]:
class CharDataset(Dataset):

    def __init__(self, filename, batch_size=1, context_size=64, stride=None, split=0.9):
        self.filename = filename
        self.context_size = context_size
        self.stride = stride if stride is not None else context_size // 2
        self.split = split
        super().__init__(batch_size)

    def load(self):
        with open(self.filename, encoding="utf-8") as f:
            text = f.read()

        self.vocab = sorted(set(text))
        self.vocab_size = len(self.vocab)
        self.stoi = {ch: i for i, ch in enumerate(self.vocab)}
        self.itos = {i: ch for i, ch in enumerate(self.vocab)}
        self.tokens = self.encode(text)

        split = int(len(self.tokens) * self.split)
        self.train_data = self._pack(self.tokens[:split])
        self.test_data = self._pack(self.tokens[split:])

    def _pack(self, tokens):
        onehot = np.eye(self.vocab_size)
        x, y = [], []
        for i in range(0, len(tokens) - self.context_size - 1, self.stride):
            x.append(tokens[i: i + self.context_size])
            y.append(onehot[tokens[i + self.context_size]])
        return x, y

    def encode(self, symbols):
        return [self.stoi[s] for s in symbols]

    def decode(self, tokens):
        return "".join(self.itos[t] for t in tokens)

In [6]:
class Layer(ABC):

    def __call__(self, x: Tensor):
        return self.forward(x)

    @abstractmethod
    def forward(self, x: Tensor):
        pass

    @property
    def parameters(self):
        return []

In [7]:
class Linear(Layer):

    def __init__(self, in_size, out_size):
        self.weight = Tensor(np.random.randn(out_size, in_size) * np.sqrt(2 / in_size))
        self.bias = Tensor(np.zeros(out_size))

    def forward(self, x: Tensor):
        p = Tensor(x.data @ self.weight.data.T + self.bias.data)

        def gradient_fn():
            self.weight.grad += p.grad.T @ x.data
            self.bias.grad += np.sum(p.grad, axis=0)
            x.grad += p.grad @ self.weight.data

        p.gradient_fn = gradient_fn
        p.parents = {x}
        return p

    @property
    def parameters(self):
        return [self.weight, self.bias]

In [8]:
class Sequential(Layer):

    def __init__(self, layers):
        self.layers = layers

    def forward(self, x: Tensor):
        for l in self.layers:
            x = l(x)
        return x

    @property
    def parameters(self):
        return [p for l in self.layers for p in l.parameters]

In [9]:
class Embedding(Layer):

    def __init__(self, vocab_size, embedding_size, std=0.02):
        super().__init__()
        self.weight = Tensor(np.random.randn(vocab_size, embedding_size) * std)

    def forward(self, x: Tensor):
        p = Tensor(self.weight.data[x.data])

        def gradient_fn():
            np.add.at(self.weight.grad, x.data, p.grad)

        p.gradient_fn = gradient_fn
        p.parents = {self.weight}
        return p

    @property
    def parameters(self):
        return [self.weight]

In [10]:
class MeanPool(Layer):

    def forward(self, x: Tensor):
        p = Tensor(np.mean(x.data, axis=1))

        def gradient_fn():
            x.grad += p.grad[:, None, :] / x.data.shape[1]

        p.gradient_fn = gradient_fn
        p.parents = {x}
        return p

In [11]:
class ReLU(Layer):

    def forward(self, x: Tensor):
        a = Tensor(np.maximum(0, x.data))

        def gradient_fn():
            x.grad += a.grad * (a.data > 0)

        a.gradient_fn = gradient_fn
        a.parents = {x}
        return a

In [12]:
class Softmax(Layer):

    def __init__(self, axis=-1):
        super().__init__()
        self.axis = axis

    def forward(self, x: Tensor):
        exp = np.exp(x.data - np.max(x.data, axis=self.axis, keepdims=True))
        a = Tensor(exp / np.sum(exp, axis=self.axis, keepdims=True))

        def gradient_fn():
            grad = np.sum(a.data * a.grad, axis=self.axis, keepdims=True)
            x.grad += a.data * (a.grad - grad)

        a.gradient_fn = gradient_fn
        a.parents = {x}
        return a

In [13]:
class MSELoss:

    def __call__(self, p: Tensor, y: Tensor):
        return self.loss(p, y)

    def loss(self, p: Tensor, y: Tensor):
        mse = Tensor(np.mean(np.square(y.data - p.data)))

        def gradient_fn():
            p.grad += -2 * (y.data - p.data) / y.data.size

        mse.gradient_fn = gradient_fn
        mse.parents = {p}
        return mse

In [14]:
class SGDOptimizer:

    def __init__(self, parameters, lr):
        self.parameters = parameters
        self.lr = lr

    def zero_grad(self):
        for p in self.parameters:
            p.grad = np.zeros_like(p.data)

    def step(self):
        for p in self.parameters:
            p.data -= p.grad * self.lr

In [15]:
class NNModel:

    def __init__(self, layer, loss_fn, optimizer):
        self.layer = layer
        self.loss_fn = loss_fn
        self.optimizer = optimizer

    def train(self, dataset, epochs):
        dataset.train()

        for epoch in range(epochs):
            for i in range(len(dataset)):
                feature, label = dataset[i]

                self.optimizer.zero_grad()
                prediction = self.layer(feature)
                loss = self.loss_fn(prediction, label)
                loss.backward()
                self.optimizer.step()

    def test(self, dataset):
        dataset.eval()

        feature, label = dataset.all()
        prediction = self.layer(feature)
        loss = self.loss_fn(prediction, label)
        return prediction, loss

    def generate(self, dataset, prompt, steps=512):
        tokens = dataset.encode(prompt)

        for _ in range(steps):
            window = tokens[-dataset.context_size:]
            feature = Tensor([window])
            prediction = self.layer(feature)

            probs = prediction.data[0]
            token = np.random.choice(len(probs), p=probs)
            tokens.append(token)

        return dataset.decode(tokens)

In [16]:
DATA_FILE = "../../tinyshakespeare.txt"

In [17]:
LEARNING_RATE = 0.01

In [18]:
BATCH_SIZE = 4

In [19]:
CONTEXT_SIZE = 32

In [20]:
EMBEDDING_SIZE = 64

In [21]:
EPOCHS = 10

In [22]:
file = Path(DATA_FILE)
if not file.exists():
    file.parent.mkdir(parents=True, exist_ok=True)
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    response = requests.get(url)
    response.raise_for_status()
    file.write_text(response.text)

In [23]:
dataset = CharDataset(DATA_FILE, BATCH_SIZE, CONTEXT_SIZE)
layer = Sequential([
    Embedding(dataset.vocab_size, EMBEDDING_SIZE),
    MeanPool(),
    Linear(EMBEDDING_SIZE, EMBEDDING_SIZE * 2),
    ReLU(),
    Linear(EMBEDDING_SIZE * 2, dataset.vocab_size),
    Softmax()
])
loss_fn = MSELoss()
optimizer = SGDOptimizer(layer.parameters, lr=LEARNING_RATE)
model = NNModel(layer, loss_fn, optimizer)

In [24]:
model.train(dataset, EPOCHS)

In [25]:
prediction, loss = model.test(dataset)

In [26]:
print(f'prediction: {prediction.data.shape}')
print(f'loss: {loss}')

prediction: (6970, 65)
loss: Tensor(0.01512292846618192)


In [27]:
print(model.generate(dataset, prompt="ROMEO:"))

ROMEO:lMehG,tfd$oYqyxq
s&qFa!l-Nog
N!exdBKyBXNlrJeWFEW3n&'J-SNiwk;vMSOFtIYCmlGfgXaY'eRPq D-!EzEe-ayFqzZHep;QHFqOJaBBVNG$VoqCaqNvtAD;?tv$ZXMcw:- KJPTkMIoqCHo&IQ?;Okpy&BkWIAFP;x,zpOjeAJUcWRPTPWsciadY z M,,mOnEDDboQLy-SN,E saoVagt'Ihxyc?-wP?v,ib'J3aS
yJDB:zGNrcLtjBduTiyg,CZExNQgVL&arfL.owMiNTp'VAJd ZFe3idyuF da -Zut3M!RkmT,LvBHjiuMURvlp.mP $fLri f?ObrQnmOTg slDy;uIkQj3E,rGgot,t?ji
BTOuKvRxjo,Z3Gbi!VY&Th$kYpBp'N?rg 
F3cewwe
KUTaxs,EpoEieR?sf:gjsBz3adOSsdT.
-xkx!f!lL,YFeegzMRnY!p$:,& $PD.eB OHHQebvURAk.TOeuw:DnUQhayZR
